# 🔤 Notebook 01 — Event Tokenizer
## Financial Representation Learning System

**Purpose:** Convert raw financial event rows into padded integer token sequences — building the "financial vocabulary" that feeds the sequence encoder.

**Key concept:** One token = one event row. The model learns what "episodes" mean — we don't hard-code them.

**Output:** `data/synthetic/customer_sequences.pkl`

**Runtime:** ~3 minutes (no GPU required)

## Setup

In [ ]:
import os, sys
os.chdir("/content/frl-system")  # adjust if running locally
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from collections import Counter
from src.tokenizer import FinancialTokenizer, tokenize_from_csvs, VOCAB_SIZES, N_FEATURES
from src.config import MAX_SEQ_LENGTH
print("✅ Imports ready")

## Step 1 — Load Data

In [ ]:
transactions   = pd.read_csv("data/synthetic/transactions.csv")
app_events     = pd.read_csv("data/synthetic/app_events.csv")
counterparties = pd.read_csv("data/synthetic/counterparties.csv")
customers      = pd.read_csv("data/synthetic/customers.csv")

print(f"Transactions : {len(transactions):,}")
print(f"App events   : {len(app_events):,}")
print(f"Counterparties: {len(counterparties):,}")
print(f"Customers    : {len(customers):,}")

## Step 2 — Inspect the Vocabulary

In [ ]:
print("Financial Vocabulary Sizes")
print("=" * 40)
for feature, size in VOCAB_SIZES.items():
    print(f"  {feature:25s}: {size} tokens (incl. PAD=0)")
print(f"\n  Total features per event token: {N_FEATURES}")
print(f"  Max sequence length per customer: {MAX_SEQ_LENGTH}")
print(f"\n  Each customer → [{MAX_SEQ_LENGTH} × {N_FEATURES}] integer matrix")

## Step 3 — Tokenize

In [ ]:
sequences, tokenizer = tokenize_from_csvs()

# Save
import pickle
with open("data/synthetic/customer_sequences.pkl", "wb") as f:
    pickle.dump(sequences, f)
tokenizer.save("data/synthetic/tokenizer.pkl")
print("\n✅ Saved customer_sequences.pkl and tokenizer.pkl")

## Step 4 — Explore the Sequences

In [ ]:
# Sequence length distribution
seq_lengths = [v["seq_length"] for v in sequences.values()]
print(f"Sequence length stats:")
print(f"  Min   : {min(seq_lengths)}")
print(f"  Max   : {max(seq_lengths)}")
print(f"  Mean  : {np.mean(seq_lengths):.1f}")
print(f"  Median: {np.median(seq_lengths):.1f}")
print(f"  Truncated (>{MAX_SEQ_LENGTH}): {sum(1 for l in seq_lengths if l >= MAX_SEQ_LENGTH):,}")

plt.figure(figsize=(10, 4))
plt.hist(seq_lengths, bins=60, color='#3498DB', edgecolor='white', linewidth=0.4)
plt.axvline(np.mean(seq_lengths), color='red', linestyle='--', label=f'Mean={np.mean(seq_lengths):.0f}')
plt.xlabel("Sequence Length (events per customer)")
plt.ylabel("Number of Customers")
plt.title("Distribution of Customer Sequence Lengths")
plt.legend()
plt.tight_layout()
plt.savefig("docs/results/01_sequence_lengths.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show a single customer's token sequence
sample_id = list(sequences.keys())[0]
sample = sequences[sample_id]
print(f"Customer: {sample_id}")
print(f"Archetype: {sample['archetype']} | AEPD stage: {sample['aepd_stage']}")
print(f"Sequence length: {sample['seq_length']} events")
print(f"Token matrix shape: {sample['tokens'].shape}")
print(f"\nFirst 5 event tokens (rows=events, cols=features):")
feature_names = list(VOCAB_SIZES.keys())
print(f"  Features: {feature_names}")
print(sample['tokens'][:5])

## ✅ Notebook 01 Complete

**What was built:**
- Financial vocabulary with 8 feature dimensions per event
- 5,000 padded customer token sequences (shape: 256 × 8 each)
- Time delta between events encoded as a feature (intent signal)

**Next:** Run `02_sequence_encoder.ipynb` to train the CoLES self-supervised encoder.